# Offline audit of a saved scoring pose

This notebook loads a retained software control with successive Vina/Vinardo
evaluations and calls public `dockingmt.audit_pose(record)`. It does not execute
Vina or interpret molecular inputs. See [the contract](pose_audit.md) and
[issue #35](https://github.com/uibcdf/dockingmt/issues/35).

The tiny control deliberately overlaps ligand and receptor. Its state labels
are declarations and its preparation remains unassessed. Internal agreement
does not establish chemical validity, score correctness or binding prediction.
Select the Python 3.14 `molsyssuite@uibcdf_3.14` kernel.

In [1]:
import json
import sys
from collections import Counter
from copy import deepcopy
from pathlib import Path

import dockingmt as dmt

assert sys.version_info[:2] == (3, 14)
assert Path(sys.prefix).name == 'molsyssuite@uibcdf_3.14'
root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'devguide/validation/data/audit/minimal_scored_pose.json').is_file()
)
record = json.loads(
    (root / 'devguide/validation/data/audit/minimal_scored_pose.json').read_text()
)
before = deepcopy(record)
baseline = dmt.audit_pose(record)
assert baseline['status'] == 'consistent'
assert len(record['scores']) == 16
assert 'vina' not in sys.modules and 'molsysviewer' not in sys.modules
print(json.dumps({
    'python': sys.version.split()[0],
    'environment': Path(sys.prefix).name,
    'scope': baseline['scope'],
    'evaluations': len(record['metadata']['scoring_history']),
    'scores': len(record['scores']),
    'checks': dict(Counter(check['status'] for check in baseline['checks'])),
}, indent=2))

{
  "python": "3.14.7",
  "environment": "molsyssuite@uibcdf_3.14",
  "scope": "saved_pose_internal_consistency",
  "evaluations": 2,
  "scores": 16,
  "checks": {
    "consistent": 378
  }
}


## Missing evidence and contradictory evidence

Remove one captured byte string without replacing it, then independently edit a
current score while retaining its original history. Contradictions take
precedence when both kinds of finding occur.

In [2]:
missing = deepcopy(record)
missing['metadata']['scoring_history'][0]['backend_artifacts']['partner'].pop(
    'content_base64'
)
changed = deepcopy(record)
changed['scores']['vina'] += 1
both = deepcopy(missing)
both['scores']['vina'] += 1
reports = {
    'retained_control': baseline,
    'missing_bytes': dmt.audit_pose(missing),
    'changed_score': dmt.audit_pose(changed),
    'missing_and_changed': dmt.audit_pose(both),
}
assert [report['status'] for report in reports.values()] == [
    'consistent', 'incomplete', 'inconsistent', 'inconsistent'
]
for name, report in reports.items():
    print(name, report['status'], dict(Counter(
        check['status'] for check in report['checks']
    )))
print(json.dumps([
    check for check in reports['changed_score']['checks']
    if check['status'] == 'inconsistent'
], indent=2))

retained_control consistent {'consistent': 378}
missing_bytes incomplete {'consistent': 377, 'incomplete': 1}
changed_score inconsistent {'consistent': 377, 'inconsistent': 1}
missing_and_changed inconsistent {'consistent': 376, 'incomplete': 1, 'inconsistent': 1}
[
  {
    "check": "evaluation_value",
    "path": "/metadata/scoring_history/0/scores/vina",
    "status": "inconsistent",
    "reason": "claims_disagree"
  }
]


## Record independence, compatibility and geometry limits

JSON roundtrips preserve declarations and audit results. Earlier records lacking
the independent name/box remain readable with incomplete audit evidence.
Coordinate shape, finiteness and length units are checked; matching coordinates
to captured PDBQT is outside this audit and would require MolSysMT.

In [3]:
roundtrip = json.loads(json.dumps(record, allow_nan=False))
assert dmt.audit_pose(roundtrip) == baseline
legacy = deepcopy(record)
for evaluation in legacy['metadata']['scoring_history']:
    evaluation.pop('score_name')
    evaluation.pop('backend_box')
assert dmt.DockingPose.from_dict(legacy).to_dict() == legacy
assert dmt.audit_pose(legacy)['status'] == 'incomplete'
geometry_edit = deepcopy(record)
geometry_edit['coordinates']['value'][0][0] += 10
assert dmt.audit_pose(geometry_edit)['status'] == 'consistent'
assert record == before
baseline['checks'][0]['status'] = 'edited_report'
assert dmt.audit_pose(record)['status'] == 'consistent'
assert 'vina' not in sys.modules and 'molsysviewer' not in sys.modules
print('JSON roundtrip: consistent; legacy evidence: incomplete')
print('Finite geometry edits need a separate molecular check.')
print('Input and reports are independent; no Vina or viewer was loaded.')

JSON roundtrip: consistent; legacy evidence: incomplete
Finite geometry edits need a separate molecular check.
Input and reports are independent; no Vina or viewer was loaded.
